# Medical RAG - Manual Retrieval Notebook

Ce notebook permet de **tester manuellement la phase Retrieval** avec un affichage lisible et clair:
- recherche hybrid (keyword + vector + RRF)
- aperçu des meilleurs chunks
- contexte enrichi (validation, summary, parent/siblings)

Test principal demandé: **`trichuris trichiura`**.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    candidates = [current] + list(current.parents)
    for c in candidates:
        if (c / "data" / "indexes" / "medical_rag.sqlite").exists() and (c / "scripts" / "retrieval").exists():
            return c
    raise RuntimeError("Impossible de localiser la racine du repo depuis le notebook.")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")


Repo root: /home/onizuka/Bureau/PFE/medical-rag-platform


In [2]:
from IPython.display import Markdown, display
import pandas as pd

from scripts.retrieval.search import SearchEngine
from scripts.retrieval.models import RetrievalFilters


def truncate(text: str | None, n: int = 160) -> str:
    t = (text or "").strip()
    return t if len(t) <= n else t[: n - 3] + "..."


def render_search_response(resp: dict):
    display(Markdown(f"## Query: `{resp['query']}`"))
    display(Markdown(f"**Mode**: `{resp['mode']}`  \
**Filtres**: `{resp.get('filters', {})}`  \
**Top results**: `{len(resp.get('top_results', []))}`"))

    top = resp.get("top_results", [])
    if not top:
        display(Markdown("Aucun résultat."))
        return

    rows = []
    for i, r in enumerate(top, start=1):
        rows.append({
            "rank": i,
            "chunk_id": r.get("chunk_id"),
            "doc_id": r.get("doc_id"),
            "chunk_type": r.get("chunk_type"),
            "source_pdf": r.get("source_pdf"),
            "page": r.get("page_number"),
            "score_hybrid": r.get("score_hybrid"),
            "score_keyword": r.get("score_keyword"),
            "score_vector": r.get("score_vector"),
            "match_reason": ", ".join(r.get("match_reason") or []),
            "preview": truncate(r.get("text_preview"), 180),
        })

    df = pd.DataFrame(rows)
    display(df)

    ctx = resp.get("context_chunks", [])
    display(Markdown(f"### Context Expansion ({len(ctx)} chunks)"))
    if not ctx:
        display(Markdown("Aucun contexte enrichi."))
        return

    ctx_rows = []
    for i, c in enumerate(ctx, start=1):
        ctx_rows.append({
            "#": i,
            "chunk_id": c.get("chunk_id"),
            "doc_id": c.get("doc_id"),
            "chunk_type": c.get("chunk_type"),
            "reason": ", ".join(c.get("match_reason") or []),
            "preview": truncate(c.get("text_preview"), 160),
        })
    display(pd.DataFrame(ctx_rows))


In [3]:
engine = SearchEngine()
print("SearchEngine initialisé.")


SearchEngine initialisé.


In [4]:
# Test manuel principal demandé
query = "trichuris trichiura"

response = engine.search(
    query=query,
    mode="hybrid",
    top_k=5,
    filters=RetrievalFilters(),
    expand_context=True,
    max_context_chunks=8,
)

render_search_response(response.to_dict())


/home/onizuka/Bureau/PFE/medical-rag-platform/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  8.39it/s]


## Query: `trichuris trichiura`

**Mode**: `hybrid`  **Filtres**: `{}`  **Top results**: `5`

,rank,chunk_id,doc_id,chunk_type,source_pdf,page,score_hybrid,score_keyword,score_vector,match_reason,preview
0,1,chk_report_exam_section_resultat_final_part_01,report,exam_section,docs/report.pdf,1,0.032522,19.623642,0.492319,"keyword_match, vector_match",RÉSULTAT FINAL: TRICHURIS TRICHIURA RÉSULTAT F...
1,2,chk_report_clinical_result_result_resultat_fin...,report,clinical_result,docs/report.pdf,1,0.032522,18.979808,0.550785,"keyword_match, vector_match",RÉSULTAT FINAL: RÉSULTAT FINAL = TRICHURIS TRI...
2,3,chk_report_12_lab_result_result_resultats_biol...,report_12,lab_result,docs/report (12).pdf,2,0.015873,NaN,0.386512,vector_match,Résultat de laboratoire: Triglycérides = 4.3 g...
3,4,chk_report_16_lab_result_result_resultats_biol...,report_16,lab_result,docs/report (16).pdf,1,0.015625,NaN,0.382416,vector_match,"Résultat de laboratoire: TSHus = 55,00 mUI/L. ..."
4,5,chk_report_10_lab_result_result_resultats_biol...,report_10,lab_result,docs/report (10).pdf,2,0.015385,NaN,0.382333,vector_match,Résultat de laboratoire: Triglycérides = 8 g/l...


### Context Expansion (8 chunks)

,#,chunk_id,doc_id,chunk_type,reason,preview
0,1,chk_report_exam_section_resultat_final_part_01,report,exam_section,"keyword_match, vector_match",RÉSULTAT FINAL: TRICHURIS TRICHIURA RÉSULTAT F...
1,2,chk_report_clinical_result_result_resultat_fin...,report,clinical_result,"keyword_match, vector_match",RÉSULTAT FINAL: RÉSULTAT FINAL = TRICHURIS TRI...
2,3,chk_report_document_summary_document_summary,report,document_summary,context_expansion:document_summary,Résumé du rapport médical. Type de document: p...
3,4,chk_report_exam_section_echantillon_biologique...,report,exam_section,context_expansion:warning,Rapport médical. Type de document: parasitolog...
4,5,chk_report_exam_section_examen_macroscopique_p...,report,exam_section,context_expansion:warning,EXAMEN MACROSCOPIQUE MUCUS: ABSENCE GLAIRE: PR...
5,6,chk_report_exam_section_examen_microscopique_p...,report,exam_section,context_expansion:warning,EXAMEN MICROSCOPIQUE LEUCOCYTES: ASSEZ NOMBREU...
6,7,chk_report_exam_section_examen_apres_enrichiss...,report,exam_section,context_expansion:warning,Rapport médical. Type de document: parasitolog...
7,8,chk_report_exam_section_examen_apres_coloratio...,report,exam_section,context_expansion:warning,Rapport médical. Type de document: parasitolog...


In [5]:
# Exemple optionnel: autre test manuel
response2 = engine.search(
    query="albumine 5 g/l",
    mode="hybrid",
    top_k=5,
    filters=RetrievalFilters(),
    expand_context=True,
    max_context_chunks=8,
)
render_search_response(response2.to_dict())


pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 2592.28it/s]


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  9.26it/s]


## Query: `albumine 5 g/l`

**Mode**: `hybrid`  **Filtres**: `{}`  **Top results**: `5`

,rank,chunk_id,doc_id,chunk_type,source_pdf,page,score_hybrid,score_keyword,score_vector,match_reason,preview
0,1,chk_report_8_lab_result_result_resultats_biolo...,report_8,lab_result,docs/report (8).pdf,1.0,0.032522,9.242239,0.677811,"keyword_match, vector_match",Résultat de laboratoire: Albumine = 5 g/l. Val...
1,2,chk_report_8_exam_section_resultats_biologique...,report_8,exam_section,docs/report (8).pdf,NaN,0.032522,9.105448,0.679856,"keyword_match, vector_match",Section médicale: Résultats biologiques. Nombr...
2,3,chk_report_29_lab_result_result_resultats_biol...,report_29,lab_result,docs/report (29).pdf,1.0,0.015873,NaN,0.664399,vector_match,Résultat de laboratoire: Albumine = 38.11 g/l....
3,4,chk_report_10_lab_result_result_resultats_biol...,report_10,lab_result,docs/report (10).pdf,1.0,0.015625,NaN,0.659281,vector_match,Résultat de laboratoire: Albumine = 8 g/l. Val...
4,5,chk_report_12_lab_result_result_resultats_biol...,report_12,lab_result,docs/report (12).pdf,1.0,0.015385,NaN,0.656274,vector_match,Résultat de laboratoire: Albumine = 40 g/l. Va...


### Context Expansion (7 chunks)

,#,chunk_id,doc_id,chunk_type,reason,preview
0,1,chk_report_8_lab_result_result_resultats_biolo...,report_8,lab_result,"keyword_match, vector_match",Résultat de laboratoire: Albumine = 5 g/l. Val...
1,2,chk_report_8_exam_section_resultats_biologique...,report_8,exam_section,"keyword_match, vector_match",Section médicale: Résultats biologiques. Nombr...
2,3,chk_report_8_document_summary_document_summary,report_8,document_summary,context_expansion:document_summary,Résumé du rapport médical. Type de document: b...
3,4,chk_report_8_exam_section_laboratory_results_p...,report_8,exam_section,context_expansion:warning,Rapport médical. Type de document: biology_rep...
4,5,chk_report_8_lab_result_result_resultats_biolo...,report_8,lab_result,context_expansion:warning,Résultat de laboratoire: AMMONIUM = 4 µg/dl. V...
5,6,chk_report_8_validation_status_validation_status,report_8,validation_status,context_expansion:validation_status,Statut administratif du rapport. Validation: u...
6,7,chk_report_8_visual_reference_visual_1_image_p...,report_8,visual_reference,context_expansion:warning,"Référence visuelle du rapport médical, page 1...."


In [6]:
# Fermer proprement le moteur après les tests
engine.close()
print("SearchEngine fermé.")


SearchEngine fermé.
